In [4]:
from transformers import Wav2Vec2FeatureExtractor, HubertModel
import torch
import torchaudio

# 加载 feature extractor 和模型（不加载 tokenizer）
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
model = HubertModel.from_pretrained("facebook/hubert-base-ls960").eval()

C:\anaconda\lib\site-packages\torch\nn\utils\weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [7]:
# 加载并处理音频
waveform, sr = torchaudio.load(r"E:\KTH\DT2119 Speech and Speaker Recognition\Project_SSR\processed\down\0a2b400e_nohash_2.wav")
if sr != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
    waveform = resampler(waveform)

# 使用 feature extractor 构造输入
inputs = feature_extractor(waveform.squeeze(0), sampling_rate=16000, return_tensors="pt")

# 模型推理
with torch.no_grad():
    outputs = model(**inputs)
    hidden_states = outputs.last_hidden_state  # shape: [1, T, D]

print("Hidden states shape:", hidden_states.shape)

Hidden states shape: torch.Size([1, 49, 768])


In [11]:
import json
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
from transformers import Wav2Vec2FeatureExtractor, HubertModel
from joblib import load  # for loading KMeans
import os

# === CONFIGURATION === #
HUBERT_MODEL = "facebook/hubert-base-ls960"
KMEANS_MODEL_PATH = "kmeans_model.pkl"
METADATA_PATH = Path(r"E:\KTH\DT2119 Speech and Speaker Recognition\Project_SSR\processed\metadata.json")
OUTPUT_DIR = Path("Project_SSR/processed_tokenized")
SAMPLE_RATE = 16000

# === LOAD MODELS === #
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(HUBERT_MODEL)
model = HubertModel.from_pretrained(HUBERT_MODEL).to(device).eval()
kmeans = load(KMEANS_MODEL_PATH)

# === UTILITY FUNCTIONS === #
def extract_tokens(wav_path: Path):
    waveform, sr = torchaudio.load(str(wav_path))
    if sr != SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
        waveform = resampler(waveform)

    inputs = feature_extractor(waveform.squeeze(0), sampling_rate=SAMPLE_RATE, return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0].cpu().numpy()  # shape: [T, D]
    tokens = kmeans.predict(hidden)
    return tokens.tolist()

def process_split(split_name, items):
    output_file = OUTPUT_DIR / f"tokens_{split_name}.jsonl"
    with open(output_file, "w") as fout:
        for item in tqdm(items, desc=f"Processing {split_name}"):
            try:
                wav_path = Path(item["path"])
                if not wav_path.is_absolute():
                    wav_path = METADATA_PATH.parent / wav_path  # relative to processed/
                tokens = extract_tokens(wav_path)
                fout.write(json.dumps({"tokens": tokens, "label": item["label"]}) + "\n")
            except Exception as e:
                print(f"[ERROR] {item['path']} failed: {e}")

# === MAIN === #
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)

    for split in ["train", "val", "test"]:
        process_split(split, metadata[split])

    print("\n✅ Done! All tokenized data saved to:", OUTPUT_DIR)

if __name__ == "__main__":
    main()


Using device: cuda


Processing test: 100%|█████████████████████████████████████████████████████████████| 3851/3851 [04:35<00:00, 13.97it/s]


✅ Done! All tokenized data saved to: Project_SSR\processed_tokenized
